# Production Deployment Tutorial

This tutorial demonstrates the complete workflow for deploying HMM models from research to production.

## Topics Covered

1. Model validation and testing
2. Artifact management and versioning
3. Production readiness checks
4. Deployment procedures
5. Monitoring and rollback strategies
6. Real-time inference integration

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import json

from imp.hmm.artifact_management import ArtifactManager
from imp.hmm.inference import HMMInference
from imp.hmm.trainer import EnhancedHMMTrainer
from imp.data.ldc_loader import LDCDataLoader
from imp.data.preprocessor import SignalPreprocessor
from imp.evaluation.evaluator import HMMEvaluator

print("✓ Imports successful")

## 2. Load Existing Artifact

Load a trained artifact that we want to deploy to production.

In [ ]:
# Initialize artifact manager
manager = ArtifactManager(artifacts_dir='../artifacts')

# Load artifact
artifact_name = 'tutorial_regime_detector'
version = '1.0.0'

artifact, metadata = manager.load_artifact(artifact_name, version)

print(f"Loaded artifact: {artifact_name} v{version}")
print(f"\nArtifact details:")
print(f"  States: {artifact.n_states}")
print(f"  Features: {artifact.n_features}")
print(f"  Training date: {metadata.get('training_date', 'N/A')}")
print(f"  Created by: {metadata.get('created_by', 'N/A')}")

## 3. Production Readiness Tests

Run comprehensive tests to ensure the artifact is ready for production.

In [ ]:
def test_artifact_production_readiness(artifact, metadata, test_data=None):
    """
    Comprehensive production readiness testing.
    """
    results = {
        'tests_passed': 0,
        'tests_failed': 0,
        'details': []
    }
    
    # Test 1: Artifact structure validation
    print("Test 1: Artifact structure validation...")
    try:
        validation_report = manager.validate_artifact(artifact)
        if validation_report['is_valid']:
            print("  ✓ PASSED")
            results['tests_passed'] += 1
            results['details'].append({'test': 'Structure validation', 'status': 'PASSED'})
        else:
            print(f"  ✗ FAILED: {validation_report['errors']}")
            results['tests_failed'] += 1
            results['details'].append({'test': 'Structure validation', 'status': 'FAILED', 'errors': validation_report['errors']})
    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        results['tests_failed'] += 1
        results['details'].append({'test': 'Structure validation', 'status': 'FAILED', 'error': str(e)})
    
    # Test 2: Inference functionality
    print("\nTest 2: Inference functionality...")
    try:
        inference = HMMInference(artifact)
        test_obs = np.random.randn(100, artifact.n_features)
        
        # Test predict_proba
        state_probs = inference.predict_proba(test_obs)
        assert state_probs.shape == (100, artifact.n_states), "Invalid state_probs shape"
        assert np.allclose(state_probs.sum(axis=1), 1.0), "Probabilities don't sum to 1"
        
        # Test predict
        states = inference.predict(test_obs)
        assert states.shape == (100,), "Invalid states shape"
        assert np.all((states >= 0) & (states < artifact.n_states)), "Invalid state values"
        
        # Test score
        score = inference.score(test_obs)
        assert np.isfinite(score), "Score is not finite"
        
        print("  ✓ PASSED")
        results['tests_passed'] += 1
        results['details'].append({'test': 'Inference functionality', 'status': 'PASSED'})
    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        results['tests_failed'] += 1
        results['details'].append({'test': 'Inference functionality', 'status': 'FAILED', 'error': str(e)})
    
    # Test 3: Metadata completeness
    print("\nTest 3: Metadata completeness...")
    required_fields = ['training_date', 'training_config', 'validation_metrics']
    missing_fields = [field for field in required_fields if field not in metadata]
    
    if not missing_fields:
        print("  ✓ PASSED")
        results['tests_passed'] += 1
        results['details'].append({'test': 'Metadata completeness', 'status': 'PASSED'})
    else:
        print(f"  ✗ FAILED: Missing fields: {missing_fields}")
        results['tests_failed'] += 1
        results['details'].append({'test': 'Metadata completeness', 'status': 'FAILED', 'missing_fields': missing_fields})
    
    # Test 4: Performance threshold (if test data provided)
    if test_data is not None:
        print("\nTest 4: Performance threshold...")
        try:
            inference = HMMInference(artifact)
            test_score = inference.score(test_data)
            
            # Get expected performance from metadata
            expected_score = metadata.get('test_performance', {}).get('log_likelihood', -np.inf)
            threshold = expected_score * 0.95  # Allow 5% degradation
            
            if test_score >= threshold:
                print(f"  ✓ PASSED (score: {test_score:.2f}, threshold: {threshold:.2f})")
                results['tests_passed'] += 1
                results['details'].append({'test': 'Performance threshold', 'status': 'PASSED', 'score': test_score})
            else:
                print(f"  ✗ FAILED (score: {test_score:.2f}, threshold: {threshold:.2f})")
                results['tests_failed'] += 1
                results['details'].append({'test': 'Performance threshold', 'status': 'FAILED', 'score': test_score, 'threshold': threshold})
        except Exception as e:
            print(f"  ✗ FAILED: {e}")
            results['tests_failed'] += 1
            results['details'].append({'test': 'Performance threshold', 'status': 'FAILED', 'error': str(e)})
    
    # Summary
    print("\n" + "="*50)
    print("PRODUCTION READINESS TEST SUMMARY")
    print("="*50)
    print(f"Tests passed: {results['tests_passed']}")
    print(f"Tests failed: {results['tests_failed']}")
    
    if results['tests_failed'] == 0:
        print("\n✓ Artifact is READY for production deployment")
    else:
        print("\n✗ Artifact is NOT READY for production deployment")
    
    return results

# Run tests
test_results = test_artifact_production_readiness(artifact, metadata)

## 4. Create Deployment Package

Prepare a complete deployment package with all necessary files and documentation.

In [ ]:
def create_deployment_package(artifact_name, version, output_dir='../deployment'):
    """
    Create a complete deployment package.
    """
    output_path = Path(output_dir) / f"{artifact_name}_v{version}"
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Load artifact
    artifact, metadata = manager.load_artifact(artifact_name, version)
    
    # 1. Copy artifact file
    artifact_source = manager.get_artifact_path(artifact_name, version)
    artifact_dest = output_path / 'model.pkl'
    import shutil
    shutil.copy(artifact_source, artifact_dest)
    print(f"✓ Copied artifact to {artifact_dest}")
    
    # 2. Save metadata
    metadata_path = output_path / 'metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    print(f"✓ Saved metadata to {metadata_path}")
    
    # 3. Create deployment documentation
    docs = f"""# Deployment Documentation

## Model Information

- **Name:** {artifact_name}
- **Version:** {version}
- **Deployment Date:** {datetime.now().isoformat()}
- **Training Date:** {metadata.get('training_date', 'N/A')}

## Model Configuration

- **States:** {artifact.n_states}
- **Features:** {artifact.n_features}
- **Library:** {metadata.get('training_config', {}).get('library', 'N/A')}
- **Covariance Type:** {metadata.get('training_config', {}).get('covariance_type', 'N/A')}

## Performance Metrics

### Validation
- Log-likelihood: {metadata.get('validation_metrics', {}).get('log_likelihood', 'N/A')}
- AIC: {metadata.get('validation_metrics', {}).get('aic', 'N/A')}
- BIC: {metadata.get('validation_metrics', {}).get('bic', 'N/A')}

### Test
- Log-likelihood: {metadata.get('test_performance', {}).get('log_likelihood', 'N/A')}

## Deployment Instructions

1. Copy `model.pkl` to production artifact directory
2. Update production configuration to use this version
3. Restart inference service
4. Monitor performance metrics

## Rollback Procedure

If issues are detected:

1. Stop inference service
2. Revert to previous version
3. Restart inference service
4. Verify rollback successful

## Monitoring

Monitor these metrics:
- Inference latency
- Log-likelihood scores
- Regime stability
- Error rates

## Contact

For issues or questions, contact: {metadata.get('created_by', 'N/A')}
"""
    
    docs_path = output_path / 'DEPLOYMENT.md'
    with open(docs_path, 'w') as f:
        f.write(docs)
    print(f"✓ Created deployment documentation at {docs_path}")
    
    # 4. Create example inference script
    inference_script = f"""#!/usr/bin/env python3
\"\"\"Example inference script for production deployment.\"\"\"\n
import numpy as np
from imp.hmm.artifact_management import ArtifactManager
from imp.hmm.inference import HMMInference

# Load artifact
manager = ArtifactManager()
artifact, metadata = manager.load_artifact('{artifact_name}', '{version}')

# Create inference engine
inference = HMMInference(artifact)

# Example: Perform inference on new data
# observations = load_your_data()  # Replace with actual data loading
# state_probs = inference.predict_proba(observations)
# states = inference.predict(observations)

print(f"Loaded model: {{artifact_name}} v{{version}}")
print(f"States: {{artifact.n_states}}")
print(f"Features: {{artifact.n_features}}")
"""
    
    script_path = output_path / 'inference_example.py'
    with open(script_path, 'w') as f:
        f.write(inference_script)
    print(f"✓ Created inference example at {script_path}")
    
    print(f"\n✓ Deployment package created at {output_path}")
    return output_path

# Create deployment package
deployment_path = create_deployment_package(artifact_name, version)

## 5. Simulate Production Inference

Test the artifact in a production-like environment.

In [ ]:
class ProductionInferenceSimulator:
    """
    Simulate production inference environment.
    """
    
    def __init__(self, artifact, metadata):
        self.artifact = artifact
        self.metadata = metadata
        self.inference = HMMInference(artifact)
        self.preprocessing = metadata.get('preprocessing', {})
        
        # Performance tracking
        self.inference_times = []
        self.scores = []
    
    def preprocess(self, observation):
        """Preprocess observation using training parameters."""
        normalization = self.preprocessing.get('normalization', {})
        if normalization:
            mean = np.array(normalization['mean'])
            std = np.array(normalization['std'])
            observation = (observation - mean) / std
        return observation
    
    def infer(self, observations):
        """Perform inference with timing."""
        import time
        
        # Preprocess
        processed = self.preprocess(observations)
        
        # Time inference
        start = time.time()
        state_probs = self.inference.predict_proba(processed)
        states = self.inference.predict(processed)
        inference_time = time.time() - start
        
        # Track performance
        self.inference_times.append(inference_time)
        score = self.inference.score(processed)
        self.scores.append(score)
        
        return {
            'state_probs': state_probs,
            'states': states,
            'inference_time': inference_time,
            'score': score
        }
    
    def get_performance_report(self):
        """Get performance statistics."""
        return {
            'avg_inference_time': np.mean(self.inference_times),
            'max_inference_time': np.max(self.inference_times),
            'min_inference_time': np.min(self.inference_times),
            'avg_score': np.mean(self.scores),
            'total_inferences': len(self.inference_times)
        }

# Create simulator
simulator = ProductionInferenceSimulator(artifact, metadata)

# Simulate multiple inference calls
print("Simulating production inference...\n")

for i in range(10):
    # Simulate batch of observations
    batch_size = np.random.randint(50, 200)
    observations = np.random.randn(batch_size, artifact.n_features)
    
    # Perform inference
    result = simulator.infer(observations)
    
    print(f"Batch {i+1}:")
    print(f"  Size: {batch_size}")
    print(f"  Inference time: {result['inference_time']*1000:.2f}ms")
    print(f"  Score: {result['score']:.2f}")

# Get performance report
print("\n" + "="*50)
print("PERFORMANCE REPORT")
print("="*50)

report = simulator.get_performance_report()
for metric, value in report.items():
    if 'time' in metric:
        print(f"{metric}: {value*1000:.2f}ms")
    else:
        print(f"{metric}: {value:.2f}")

## 6. Create Monitoring Dashboard

Set up monitoring for production deployment.

In [ ]:
import matplotlib.pyplot as plt

def create_monitoring_dashboard(simulator):
    """
    Create monitoring dashboard for production model.
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: Inference times
    axes[0, 0].plot(simulator.inference_times)
    axes[0, 0].axhline(y=np.mean(simulator.inference_times), color='r', linestyle='--', label='Mean')
    axes[0, 0].set_title('Inference Times')
    axes[0, 0].set_xlabel('Batch')
    axes[0, 0].set_ylabel('Time (seconds)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Scores
    axes[0, 1].plot(simulator.scores)
    axes[0, 1].axhline(y=np.mean(simulator.scores), color='r', linestyle='--', label='Mean')
    axes[0, 1].set_title('Log-Likelihood Scores')
    axes[0, 1].set_xlabel('Batch')
    axes[0, 1].set_ylabel('Score')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Inference time distribution
    axes[1, 0].hist(simulator.inference_times, bins=20, edgecolor='black')
    axes[1, 0].set_title('Inference Time Distribution')
    axes[1, 0].set_xlabel('Time (seconds)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Score distribution
    axes[1, 1].hist(simulator.scores, bins=20, edgecolor='black')
    axes[1, 1].set_title('Score Distribution')
    axes[1, 1].set_xlabel('Score')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Create dashboard
dashboard = create_monitoring_dashboard(simulator)
plt.show()

print("\n✓ Monitoring dashboard created")

## 7. Summary and Next Steps

### Deployment Checklist

- ✓ Artifact loaded and validated
- ✓ Production readiness tests passed
- ✓ Deployment package created
- ✓ Production inference tested
- ✓ Monitoring dashboard prepared

### Next Steps for Production Deployment

1. **Review deployment package** in `../deployment/` directory
2. **Copy artifact** to production environment
3. **Update production configuration** to use new version
4. **Deploy with canary strategy** (10% → 50% → 100% traffic)
5. **Monitor performance metrics** continuously
6. **Set up alerts** for performance degradation
7. **Document rollback procedure** for team

### Monitoring Recommendations

Monitor these metrics in production:

- **Inference latency**: Should stay below 100ms for real-time trading
- **Log-likelihood scores**: Should remain stable (within 10% of test performance)
- **Regime stability**: Check for excessive regime switching
- **Error rates**: Monitor for inference failures
- **Memory usage**: Track for memory leaks

### Rollback Triggers

Rollback to previous version if:

- Inference latency exceeds 200ms
- Log-likelihood drops by more than 20%
- Error rate exceeds 1%
- Memory usage grows unbounded

### Resources

- [Integration Examples](../py/docs/INTEGRATION_EXAMPLES.md)
- [Best Practices](../py/docs/BEST_PRACTICES.md)
- [Troubleshooting Guide](../py/docs/TROUBLESHOOTING.md)